# The Merit-Order Model - an interactive notebook
**Course materials for *32961 - Sustainable Energy Economics (SusEE)***

This notebook lets you build the merit-order model of Section 2.4.1 yourself, step by step, and reproduces several of the lecture notes.

**What you will do**

| Section | Topic | Lecture notes |
|---|---|---|
| 2 | The plant fleet | Table 2-11 |
| 3 | Marginal cost and the carbon price | Formula 19 |
| 4 | The merit order and the market price | Formula 20 |
| 5 | The supply curve | Figure 2-10 |
| 6 | Inframarginal rents | Formula 21 |
| 7 | The merit-order effect of renewables | Figure 2-11 |
| 8 | Prices with and without renewables | Table 2-12 |
| 9 | The carbon price and the fuel switch | Figure 2-12 |
| 10 | Market value and self-cannibalisation | Formulas 22 and 23 |
| 11 | Market splitting under a transmission constraint | (Figure 2-17) |
| 12 | Long-term investment | (Figure 2-20) |

**One important hint before you start.** You do not need an optimisation solver for any of this. The merit-order model is solved by **sorting**: line the plants up from cheapest to most expensive, add up their capacities, and see which plant serves the last megawatt of demand. That plant sets the price.

The formal optimisation problem behind the same model (Section 2.4.1.3) is implemented separately in **GAMS** in a separate file.

## 1  Setup

We only need three standard libraries. To install them, please run **<kbd>pip install pandas matplotlib ipywidgets</kbd>** in the terminal.

In [ ]:
# If something is missing, run once:
# pip install pandas matplotlib ipywidgets

import pandas as pd                 # tables
import numpy as np                  # numbers
import matplotlib.pyplot as plt     # plots
import matplotlib.patheffects as pe # the drop shadow on the legend box

try:
    from ipywidgets import interact, IntSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not installed - Section 11 will show a static example instead.\n"
          "To get the sliders, run:  pip install ipywidgets")

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

def interactive(fn, static_example=None, **slider_specs):
    """Show sliders for fn if ipywidgets is available, otherwise run one static example.
    Each slider is given as  name=(min, max, step, start_value, label)."""
    if HAS_WIDGETS:
        widgets = {name: IntSlider(min=lo, max=hi, step=st, value=val,
                                   description=lab,
                                   style={"description_width": "135px"},
                                   continuous_update=False)   # redraw on release, not while dragging
                   for name, (lo, hi, st, val, lab) in slider_specs.items()}
        interact(fn, **widgets)
    else:
        print("(ipywidgets not installed - showing one static example instead)\n")
        fn(**(static_example or {}))

print("Ready.")

## 2  The plant fleet (Table 2-11)

Nine power plants. For each one we store:

- `capacity` - how many MW it can produce at most,
- `c_var0` - the part of its marginal cost that has **nothing** to do with CO₂ (fuel divided by efficiency, plus other variable costs),
- `ef` - the **emission factor**: how many tonnes of CO₂ it emits per MWh of electricity,
- `colour` - only for the plots.

Splitting the marginal cost into `c_var0` and `ef` is what lets us change the carbon price later (Section 3). The numbers are identical to the GAMS base model.

In [ ]:
# One row per power plant. The index is the short name used in the code.
TECH = pd.DataFrame({
    "label":    ["Run-of-river", "Nuclear", "Lignite", "Gas CCGT", "Hard coal 1",
                 "Gas 1", "Hard coal 2", "Biomass", "Gas 2"],
    "capacity": [50, 1000, 800, 700, 750, 200, 650, 20, 150],          # MW
    "c_var0":   [0.00, 6.06, 12.25, 34.11, 32.15, 49.00, 42.61, 66.82, 67.50],  # €/MWh, without CO₂
    "ef":       [0.00, 0.00, 1.05, 0.35, 0.80, 0.50, 0.80, 0.00, 0.50],         # t CO₂ / MWh_el
    "colour":   ["#3182bd", "#c724b1", "#7b5141", "#fdae6b", "#333333",
                 "#e6550d", "#7a7a7a", "#31a354", "#fb9a29"],
}, index=["runOfRiver", "nuclear", "lignite", "combinedCycle", "hardcoal_1",
          "gas_1", "hardcoal_2", "biomass", "gas_2"])

# Demand in the six time steps of the base model, and the length of a time step
DEMAND = pd.Series([3000, 3500, 4200, 3900, 4000, 3600],
                   index=[f"t{i+1}" for i in range(6)], name="demand_MW")
DT = 4              # hours per time step
CO2_BASE = 25.0     # €/t - the carbon price of the base model

# Colours for the renewables we add in Section 7
VRE_COLOUR = {"Wind": "#74c476", "PV": "#fdd835"}

TECH[["label", "capacity", "c_var0", "ef"]]

## 3  Marginal cost and the carbon price (Formula 19)

Formula 19 of the lecture notes says

$$c_i = \frac{f_i}{\eta_i} + o_i + ef_i \cdot p^{CO_2}$$

The first two terms are what we stored as `c_var0`. So all we have to add is the CO₂ part:

$$c_i = \texttt{c\_var0}_i + ef_i \cdot p^{CO_2}$$

Notice what this does: a plant with a **high emission factor gets more expensive faster** when the carbon price rises. That is the whole mechanism behind the fuel switch in Section 9.

In [ ]:
def marginal_cost(co2_price=CO2_BASE):
    """Marginal cost of every plant in €/MWh, for a given carbon price in €/t.
    This is Formula 18."""
    return TECH["c_var0"] + TECH["ef"] * co2_price

# At the carbon price of the base model we get exactly the numbers of Table 2-10
check = pd.DataFrame({
    "label": TECH["label"],
    "marginal cost @ 25 €/t": marginal_cost(CO2_BASE),
    "marginal cost @ 80 €/t": marginal_cost(80),
})
check.sort_values("marginal cost @ 25 €/t")

## 4  The merit order and the market price (Formula 20)

Formula 20 of the lecture notes defines the price-setting plant and the resulting market price as

$$
p_t = c_{m(t)},
\qquad
m(t)=\min\left\{k \,\middle|\, \sum_{i=1}^{k} y_i^{\max}>D_t\right\}
$$

The plants are indexed in ascending order of marginal cost. The model therefore follows four steps.

1. **Sort** the plants by marginal cost, beginning with the cheapest plant. This produces the merit order.
2. **Add up** their capacities along this order. This gives the cumulative capacity $\sum_{i=1}^{k}y_i^{\max}$.
3. Identify the first plant $m(t)$ for which cumulative capacity exceeds demand $D_t$. This is the cheapest plant that still has spare capacity.
4. Set the market price $p_t$ equal to the marginal cost $c_{m(t)}$ of this plant.

The notebook implements these steps with `sort_values()` and `cumsum()`. No optimisation solver is required.

The strict inequality $>$ is important when demand lies exactly on a capacity boundary. In this case, all plants up to that boundary are fully used and the next plant in the merit order would supply one additional MWh. Its marginal cost therefore determines the market price. Time step t2 is such a boundary case.

In [ ]:
def merit_order(co2_price=CO2_BASE, wind=0.0, pv=0.0):
    """Return the merit order as a table: plants sorted by marginal cost,
    with their cumulative capacity. Wind and PV are added at ~0 €/MWh."""
    rows = pd.DataFrame({
        "label":    TECH["label"],
        "capacity": TECH["capacity"].astype(float),
        "mc":       marginal_cost(co2_price),
        "colour":   TECH["colour"],
    })
    # add the renewables (only if a capacity was given)
    extra = []
    if wind > 0:
        extra.append(pd.DataFrame({"label": ["Wind"], "capacity": [float(wind)],
                                   "mc": [0.0], "colour": [VRE_COLOUR["Wind"]]}, index=["wind"]))
    if pv > 0:
        extra.append(pd.DataFrame({"label": ["PV"], "capacity": [float(pv)],
                                   "mc": [0.001], "colour": [VRE_COLOUR["PV"]]}, index=["pv"]))
    if extra:
        rows = pd.concat([rows] + extra)

    rows = rows.sort_values("mc", kind="stable")          # step 1: sort
    rows["cum_before"] = rows["capacity"].cumsum() - rows["capacity"]
    rows["cum_after"]  = rows["capacity"].cumsum()        # step 2: cumulate
    return rows


def market_price(demand, co2_price=CO2_BASE, wind=0.0, pv=0.0):
    """Return (price, name of the price-setting plant) for a given demand in MW.
    This is Formula 19.

    The price is the marginal cost of the cheapest plant that still has SPARE
    capacity, that is, the cost of serving one MORE MWh. That is exactly what
    the dual variable of the demand constraint means, which is why this rule
    reproduces the GAMS model to the cent (Section 12). The strict ">" matters
    only when demand lands exactly on a block boundary, as it does in t2."""
    mo = merit_order(co2_price, wind, pv)
    # step 3: the cheapest plant that is not already fully used
    spare = mo[mo["cum_after"] > demand + 1e-9]
    if len(spare) == 0:
        return np.nan, "demand at or above total capacity"
    row = spare.iloc[0]
    return row["mc"], row["label"]                        # step 4: its marginal cost


# The merit order of the base model - compare this with Table 2-10
mo = merit_order()
display(mo[["label", "capacity", "mc", "cum_after"]])

# The price in every time step
for t, d in DEMAND.items():
    p, plant = market_price(d)
    print(f"{t}: demand {d:5.0f} MW  ->  price {p:6.2f} €/MWh   (set by {plant})")

## 5  Figure 2-10: the supply curve

Each plant is drawn as a block: **as wide as its capacity**, **as high as its marginal cost**. Stacking the blocks in merit order from left to right traces out the supply curve. Demand is the vertical dashed line, and where it hits the curve we read off the price.

The function below is the one that produced Figure 2-10 in the lecture notes, so you do not have to build the figure separately: just run the cell.

In [ ]:
def figure_2_9(demand=4200, co2_price=CO2_BASE, wind=0.0, pv=0.0,
               ax=None, ymax=None, save_as=None):
    """Draw the merit-order supply curve with demand and price (Figure 2-9).
    ymax=None scales the y-axis automatically, so the picture still works
    at a high carbon price."""
    mo = merit_order(co2_price, wind, pv)
    if ymax is None:
        ymax = max(105, mo["mc"].max() * 1.30)
    price, plant = market_price(demand, co2_price, wind, pv)
    gray, colour_price = "#666666", "#e6550d"

    if ax is None:
        fig, ax = plt.subplots(figsize=(9.5, 5.2))
    else:
        fig = ax.figure

    # one bar per plant: width = capacity, height = marginal cost
    for _, r in mo.iterrows():
        ax.bar(r["cum_before"], r["mc"], width=r["capacity"], align="edge",
               color=r["colour"], edgecolor="white", linewidth=0.7,
               label=r["label"], zorder=3)

    # demand: a vertical line (demand does not react to the price)
    ax.axvline(demand, color=gray, linestyle="--", linewidth=1.4, zorder=4)
    ax.text(demand - 70, ymax * 0.962, "Market demand D", color=gray,
            fontsize=9.5, ha="right", va="top")

    # price: a horizontal line at the marginal cost of the price-setting plant
    if np.isfinite(price):
        ax.plot([0, demand], [price, price], color=colour_price,
                linestyle=":", linewidth=1.9, zorder=4)
        ax.text(demand * 0.63, price - 3,
                f"Market price p = {price:.2f} €/MWh\n({plant} is the marginal plant)",
                color=colour_price, fontsize=9.5, va="top", ha="left")

    total = mo["cum_after"].max()
    ax.set_xlim(0, max(total, demand) * 1.01)
    ax.set_ylim(0, ymax)
    ax.set_xlabel("Cumulative available capacity (MW)", fontsize=10)
    ax.set_ylabel("Marginal cost, price (€/MWh)", fontsize=10)
    ax.grid(False)
    ax.set_axisbelow(True)
    ax.set_facecolor("white")

    # a flat 3-column legend, so it sits above the price line instead of on top of it
    legend = ax.legend(loc="upper left", bbox_to_anchor=(0.015, 0.975), frameon=True,
                       fontsize=8.5, borderaxespad=0.0, fancybox=True,
                       markerfirst=False, handletextpad=0.7, ncol=3, columnspacing=1.2)
    frame = legend.get_frame()
    frame.set_facecolor("white"); frame.set_edgecolor("#bdbdbd")
    frame.set_linewidth(0.7); frame.set_alpha(0.96)
    if callable(getattr(frame, "set_boxstyle", None)):
        set_boxstyle = getattr(frame, "set_boxstyle", None)
        if callable(set_boxstyle):
            set_boxstyle("round,pad=0.35,rounding_size=0.18")
    frame.set_path_effects([pe.SimplePatchShadow(offset=(1.5, -1.5), alpha=0.18,
                                                 shadow_rgbFace="#999999"), pe.Normal()])
    for txt in legend.get_texts():
        txt.set_horizontalalignment("right")
    if getattr(legend, "_legend_box", None) is not None:
        box = getattr(legend, "_legend_box", None)
        if box is not None:
            setattr(box, "align", "right")

    fig.tight_layout()
    if save_as:
        fig.savefig(save_as, bbox_inches="tight")
        print("saved to", save_as)
    return ax


figure_2_9(demand=4200)
plt.show()

### Move the demand line yourself

Drag the **demand** slider and watch which block the line lands in: that block is the price-setting plant. Drag the **carbon price** slider and watch the blocks change height, and eventually change places.

Try demand levels of 3,299 and 3,301 MW. You can also click directly on the number displayed in the input box. An increase in demand of only 2 MW raises the market price from 52.15 to 61.50 €/MWh because it exhausts the available capacity of Hard coal 1 and causes Gas 1 to become the marginal plant. This illustrates how sensitive electricity prices can be to small changes in demand when the market is close to a capacity boundary. The resulting price jump is not a model error but a consequence of the step-shaped supply curve. Such abrupt changes help explain why electricity prices are generally more volatile than the prices of storable goods.

In [ ]:
def show_supply_curve(demand=4200, co2_price=25):
    figure_2_9(demand=demand, co2_price=co2_price)   # the y-axis scales itself
    plt.show()
    p, plant = market_price(demand, co2_price)
    if np.isfinite(p):
        print(f"Price {p:.2f} €/MWh, set by {plant}.")
    else:
        print("Demand exceeds the total capacity of 4320 MW - the market cannot clear.")

interactive(show_supply_curve,
            static_example=dict(demand=3500, co2_price=25),
            demand=(500, 5000, 100, 4200, "demand (MW)"),
            co2_price=(0, 150, 5, 25, "CO₂ price (€/t)"))

**Read the figure.** Demand of 4200 MW cuts the curve inside the Gas 2 block, so Gas 2 is the marginal plant and the price is 80.00 €/MWh. Everything to the left of the demand line runs at full capacity; everything to the right is more expensive than the price and stays off.

Try changing `demand` to 3000 or 3500 and watch which plant sets the price.

## 6  Inframarginal rents (Formula 21)

Every plant that runs is paid the **same** market price, not its own bid (this is called *uniform pricing*). So a cheap plant pockets the difference between the price and its own marginal cost:

$$R_{i,t} = (p_t - c_i) \cdot y_{i,t} \cdot \Delta t$$

This is not a windfall. It is how a plant recovers the fixed costs that the price of the marginal plant does not cover (Section 2.4.3). The marginal plant itself earns nothing: for it, price = marginal cost.

In [ ]:
def dispatch_and_rents(demand, co2_price=CO2_BASE, wind=0.0, pv=0.0):
    """How much does each plant produce, and what rent does it earn?"""
    mo = merit_order(co2_price, wind, pv)
    price, _ = market_price(demand, co2_price, wind, pv)

    # a plant produces the part of its block that lies left of the demand line
    output = (demand - mo["cum_before"]).clip(lower=0)
    output = pd.concat([output, mo["capacity"]], axis=1).min(axis=1)

    out = pd.DataFrame({
        "label":  mo["label"],
        "output_MW": output,
        "mc":     mo["mc"],
        "rent_€": (price - mo["mc"]) * output * DT,
    })
    return out[out["output_MW"] > 0], price


table, price = dispatch_and_rents(4200)
print(f"Market price: {price:.2f} €/MWh   (time step = {DT} h)\n")
display(table)
print("The lignite plant in this time step earns an inframarginal rent of 132,80 € "
      "((80.00 - 38.50) x 800 MW x 4 h), while Gas 2 produces 30 MW and earns no inframarginal rent "
      "because its marginal cost is equal to the market price.")

## 7  The merit-order effect of renewables (Figure 2-11)

Wind and PV have a marginal cost of about **zero**, so they join the merit order at the very left. They do not change the order of the conventional plants: they **push the whole conventional stack to the right** by the amount they feed in.

The result is the *merit-order effect*: demand has not moved, but the supply curve has, so a cheaper plant now serves the last megawatt and **the price falls**.

Figure 2-11 draws this as two **step curves** rather than blocks. The reason is worth understanding: wind and PV sit at a height of ~0 €/MWh, so as coloured blocks they would be invisible. The green double arrow shows where they are.

In [ ]:
def _step_curve(mo, offset=0.0):
    """Turn a merit-order table into the x/y points of a step curve."""
    xs, ys = [], []
    x = offset
    for _, r in mo.iterrows():
        xs += [x, x + r["capacity"]]
        ys += [r["mc"], r["mc"]]
        x += r["capacity"]
    return xs, ys


def figure_2_10(demand=4200, wind=1200, pv=800, co2_price=CO2_BASE, ymax=None, save_as=None):
    """Draw the merit-order effect: the supply curve with and without renewables
    (Figure 2-10)."""
    res = wind + pv
    conv = merit_order(co2_price)                   # conventional plants only
    if ymax is None:
        ymax = max(105, conv["mc"].max() * 1.30)    # keep headroom at high carbon prices
    p_without, plant_without = market_price(demand, co2_price)
    p_with,    plant_with    = market_price(demand, co2_price, wind, pv)

    x0, y0 = _step_curve(conv, 0.0)                 # without renewables
    x1, y1 = _step_curve(conv, res)                 # shifted right by the RES feed-in

    gray, c_base, c_shift, c_res = "#666666", "#08519c", "#e6550d", "#74c476"
    fig, ax = plt.subplots(figsize=(9.5, 5.2))

    ax.plot(x0, y0, color=gray, linestyle="--", linewidth=1.8, zorder=3)
    ax.plot(x1, y1, color=c_base, linestyle="-", linewidth=2.4, zorder=4)

    # the renewables: zero marginal cost, so they lie flat on the axis -> show as a span.
    # (only if there are any: a zero-length arrow would break the layout)
    if res > 0:
        ax.annotate("", xy=(0, 6.5), xytext=(res, 6.5),
                    arrowprops=dict(arrowstyle="<->", color=c_res, linewidth=2.0))
        ax.text(res / 2, 8.0, f"Wind + PV: {res:.0f} MW at a marginal cost of about zero",
                color="#2f7d43", fontsize=9.5, ha="center", va="bottom")
        ax.axvline(res, color=c_res, linestyle=":", linewidth=1.4, zorder=2)

    ax.axvline(demand, color=gray, linestyle="--", linewidth=1.4, zorder=3)
    ax.text(demand - 90, ymax * 0.962, "Market demand D",
            color=gray, fontsize=9.5, ha="right", va="top")

    ax.plot([0, demand], [p_without, p_without], color=gray, linestyle=":", linewidth=1.6, zorder=3)
    ax.plot([0, demand], [p_with, p_with], color=c_shift, linestyle=":", linewidth=1.9, zorder=4)
    ax.plot([demand], [p_without], marker="o", color=gray, markersize=7, zorder=6)
    ax.plot([demand], [p_with], marker="o", color=c_shift, markersize=7, zorder=6)

    # the arrow that measures the merit-order effect
    # (only if the price actually moved: a zero-length arrow would break the layout)
    xa = demand + 350
    if abs(p_with - p_without) > 1e-9:
        ann = ax.annotate("", xy=(xa, p_with), xytext=(xa, p_without),
                          arrowprops=dict(arrowstyle="->", color=c_shift, linewidth=3.0),
                          zorder=11)
        if ann.arrow_patch is not None:
            ann.arrow_patch.set_zorder(10)
        ax.text(xa + 30, (p_with + p_without) / 2 + 5,
            f"Merit-order effect\n{p_with - p_without:.2f} €/MWh",
            color=c_shift, fontsize=9.5, ha="left", va="center")
    else:
        ax.text(xa + 60, p_without, "no merit-order effect\nat this demand",
                color=gray, fontsize=9.5, ha="left", va="center")

    if abs(p_with - p_without) > 1e-9:
        ax.text(120, p_without + 1.5, f"p = {p_without:.2f} €/MWh ({plant_without} marginal)",
                color=gray, fontsize=9.5, ha="left", va="bottom")
        ax.text(120, p_with + 1.5, f"p = {p_with:.2f} €/MWh ({plant_with} marginal)",
                color=c_shift, fontsize=9.5, ha="left", va="bottom")
    else:
        ax.text(120, p_without + 1.5, f"p = {p_without:.2f} €/MWh ({plant_without} marginal)",
                color=gray, fontsize=9.5, ha="left", va="bottom")

    ax.set_xlim(0, conv["cum_after"].max() + res + 1130)
    ax.set_ylim(0, ymax)
    ax.set_xlabel("Cumulative available capacity (MW)", fontsize=10)
    ax.set_ylabel("Marginal cost, price (€/MWh)", fontsize=10)
    ax.grid(False)
    ax.set_axisbelow(True)
    ax.set_facecolor("white")

    from matplotlib.lines import Line2D
    handles = [Line2D([0], [0], color=gray, linestyle="--", linewidth=1.8,
                      label="Supply curve without renewables"),
               Line2D([0], [0], color=c_base, linestyle="-", linewidth=2.4,
                      label="Supply curve with wind and PV")]
    legend = ax.legend(handles=handles, loc="upper right", bbox_to_anchor=(0.985, 0.975),
                       frameon=True, fontsize=9, borderaxespad=0.0, fancybox=True,
                       markerfirst=False, handletextpad=0.8)
    frame = legend.get_frame()
    frame.set_facecolor("white"); frame.set_edgecolor("#bdbdbd")
    frame.set_linewidth(0.7); frame.set_alpha(0.96)
    if callable(getattr(frame, "set_boxstyle", None)):
        set_boxstyle = getattr(frame, "set_boxstyle", None)
        if callable(set_boxstyle):
            set_boxstyle("round,pad=0.35,rounding_size=0.18")
    frame.set_path_effects([pe.SimplePatchShadow(offset=(1.5, -1.5), alpha=0.18,
                                                 shadow_rgbFace="#999999"), pe.Normal()])
    for txt in legend.get_texts():
        txt.set_horizontalalignment("right")
    if getattr(legend, "_legend_box", None) is not None:
        if hasattr(legend, "set_alignment"):
            legend.set_alignment("right")

    fig.tight_layout()
    if save_as:
        fig.savefig(save_as, bbox_inches="tight")
        print("saved to", save_as)
    return ax

figure_2_10(demand=4200, wind=1200, pv=800)
plt.show()

### Shift the supply curve yourself

This is the slider to spend time with. Push **wind** and **PV** up and watch the blue curve slide to the right while the grey one stays put. The gap between the two price dots is the merit-order effect, and it is printed underneath.

Two things to look for:

- Raise the renewables slowly from 0. The price does **not** fall smoothly, it falls in **jumps**, and between the jumps nothing happens at all. The jumps occur exactly when the intersection crosses from one block into the next.
- Set `demand` to 3000 and raise wind to 450, then to 500. Nothing, then a large drop. Look at the cumulative capacity column of Table 2-12 to see why.

In [ ]:
def show_mo_effect(wind=1200, pv=800, demand=4200, co2_price=25):
    figure_2_10(demand=demand, wind=wind, pv=pv, co2_price=co2_price)
    plt.show()
    p0, plant0 = market_price(demand, co2_price)
    p1, plant1 = market_price(demand, co2_price, wind, pv)
    print(f"without renewables : {p0:6.2f} €/MWh   (marginal: {plant0})")
    print(f"with renewables    : {p1:6.2f} €/MWh   (marginal: {plant1})")
    print(f"merit-order effect : {p1 - p0:+6.2f} €/MWh")

interactive(show_mo_effect,
            static_example=dict(wind=1200, pv=800, demand=4200, co2_price=25),
            wind=(0, 3000, 50, 1200, "wind (MW)"),
            pv=(0, 3000, 50, 800, "PV (MW)"),
            demand=(500, 5000, 100, 4200, "demand (MW)"),
            co2_price=(0, 150, 5, 25, "CO₂ price (€/tCO₂)"))

## 8  Table 2-12, prices with and without renewables

Table 2-12 applies the same 2,000 MW of renewable feed-in, consisting of 1,200 MW of wind and 800 MW of PV, to all six time steps. Examine the **difference** column. Although renewable generation is identical in every time step, the resulting price reduction ranges from 19.75 to 46.09 €/MWh.

### Why the effect differs across time steps

The merit-order effect varies because demand intersects the step-shaped supply curve at a different point in each time step. In t1, the market price falls from 52.15 to 6.06 €/MWh and the price-setting plant changes from Hard coal 1 to Nuclear. The renewable feed-in therefore moves the intersection across several steps of the supply curve. In t4, the price falls from 62.61 to 42.86 €/MWh and the price-setting plant changes only from Hard coal 2 to Gas CCGT. The size of the merit-order effect therefore depends on the level of demand and the shape of the supply curve as well as on the amount of renewable generation.

In [ ]:
def price_table(wind=1200, pv=800, co2_price=CO2_BASE):
    """Compare the price in every time step, with and without renewables (Table 2-11)."""
    rows = []
    for t, d in DEMAND.items():
        p0, _ = market_price(d, co2_price)
        p1, _ = market_price(d, co2_price, wind, pv)
        rows.append({"time step": t, "demand_MW": d,
                     "price without RES": p0, "price with RES": p1,
                     "difference": p1 - p0})
    tab = pd.DataFrame(rows).set_index("time step")
    tab.loc["Average"] = tab.mean()
    return tab

price_table()

### The same table as a picture

The bars show the price in each time step with and without renewables. Move the sliders and watch **both** things at once: how far each bar drops, and how unevenly.

In [ ]:
def plot_prices_over_day(wind=1200, pv=800, co2_price=25):
       tab = price_table(wind, pv, co2_price).drop(index="Average")
       x = np.arange(len(tab))

       fig, ax = plt.subplots(figsize=(9.5, 5.0))
       ax.bar(x - 0.2, tab["price without RES"], width=0.38, color="#9e9e9e",
                 edgecolor="white", label="without renewables", zorder=3)
       ax.bar(x + 0.2, tab["price with RES"], width=0.38, color="#08519c",
                 edgecolor="white", label="with wind and PV", zorder=3)

       # the drop, written above each pair
       for xi, (p0, p1) in enumerate(zip(tab["price without RES"], tab["price with RES"])):
              ax.annotate(f"{p1 - p0:+.1f}", xy=(xi, max(p0, p1) + 1.5), ha="center",
                                   va="bottom", fontsize=9, color="#e6550d")

       ax.set_xticks(x)
       ax.set_xticklabels(tab.index)
       ax.set_xlabel("Time step", fontsize=10)
       ax.set_ylabel("Market price (€/MWh)", fontsize=10)
       ax.set_ylim(0, max(tab[["price without RES", "price with RES"]].max()) * 1.22)
       ax.set_title(f"Prices with {wind:.0f} MW wind and {pv:.0f} MW PV "
                             f"({co2_price:.0f} €/tCO₂)", fontsize=11)
       ax.grid(False)
       ax.set_axisbelow(True)
       ax.set_facecolor("white")

       # legend style like in the other figures
       legend = ax.legend(loc="upper right", bbox_to_anchor=(0.985, 0.975),
                                      frameon=True, fontsize=9, borderaxespad=0.0, fancybox=True,
                                      markerfirst=False, handletextpad=0.8)
       frame = legend.get_frame()
       frame.set_facecolor("white")
       frame.set_edgecolor("#bdbdbd")
       frame.set_linewidth(0.7)
       frame.set_alpha(0.96)
       if callable(getattr(frame, "set_boxstyle", None)):
                set_boxstyle = getattr(frame, "set_boxstyle", None)
                if callable(set_boxstyle):
                             set_boxstyle("round,pad=0.35,rounding_size=0.18")
       frame.set_path_effects([
              pe.SimplePatchShadow(offset=(1.5, -1.5), alpha=0.18, shadow_rgbFace="#999999"),
              pe.Normal()
       ])
       for txt in legend.get_texts():
              txt.set_horizontalalignment("right")
       if getattr(legend, "_legend_box", None) is not None and hasattr(legend, "set_alignment"):
              legend.set_alignment("right")

       fig.tight_layout()
       plt.show()

       print(f"average price without renewables: {tab['price without RES'].mean():.2f} €/MWh")
       print(f"average price with renewables   : {tab['price with RES'].mean():.2f} €/MWh")


interactive(plot_prices_over_day,
                     static_example=dict(wind=1200, pv=800, co2_price=25),
                     wind=(0, 3000, 50, 1200, "wind (MW)"),
                     pv=(0, 3000, 50, 800, "PV (MW)"),
                     co2_price=(0, 150, 5, 25, "CO₂ price (€/tCO₂)"))

## 9  The carbon price and the fuel switch

A carbon price does **not** raise all plants equally. By Formula 19 it raises each plant's cost by `ef x carbon price`, so the dirtiest plants get more expensive fastest. Push it far enough and two plants **swap places** in the merit order.

Lignite (`ef` = 1.05) and gas CCGT (`ef` = 0.35) are the classic pair. Set their marginal costs equal and solve for the carbon price:

$$12.25 + 1.05\,p^{\mathrm{CO_2}} = 34.11 + 0.35\,p^{\mathrm{CO_2}} \;\Longrightarrow\; p^{\mathrm{CO_2}} = \frac{34.11-12.25}{1.05-0.35}$$

This **fuel switch** is the fastest way carbon pricing cuts emissions: no new plant is needed, only a different order in the existing fleet.

In [ ]:
# The switching price, straight from the two rows of the table
a, b = TECH.loc["lignite"], TECH.loc["combinedCycle"]
p_switch = (b["c_var0"] - a["c_var0"]) / (a["ef"] - b["ef"])
print(f"Fuel-switch carbon price, lignite vs gas CCGT: {p_switch:.2f} €/t\n")

for co2 in [0, 25, p_switch, 60, 80]:
    mc = marginal_cost(co2)
    diff = mc["combinedCycle"] - mc["lignite"]
    if abs(diff) < 1e-6:
        verdict = "equal - this is the switching point"
    else:
        verdict = "cheaper: " + ("gas CCGT" if diff < 0 else "lignite")
    print(f"  CO₂ = {co2:6.2f} €/tCO₂ ->  lignite {mc['lignite']:6.2f} | "
          f"gas CCGT {mc['combinedCycle']:6.2f}  ->  {verdict}")

print("\nFor reference: EU ETS allowance prices moved roughly between 60 and 80 €/t\n"
      "in 2025 (European Commission, 2025) - well above the switching point.")

In [ ]:
# What the whole merit order looks like at a high carbon price
fig, axes = plt.subplots(2, 1, figsize=(9.5, 9.4))

figure_2_9(demand=4000, co2_price=25, ax=axes[0], ymax=105)
axes[0].set_title("Carbon price 25 €/tCO$_2$ (the base model)", fontsize=11)

figure_2_9(demand=4000, co2_price=80, ax=axes[1], ymax=155)
axes[1].set_title("Carbon price 80 €/tCO$_2$ (lignite has moved to the right)", fontsize=11)

# Move orange price label above the orange price line in each panel
for ax in axes:
    # find the orange horizontal price line
    price_lines = [ln for ln in ax.lines if ln.get_color() == "#e6550d" and ln.get_linestyle() == ":"]
    if not price_lines:
        continue
    y_price = price_lines[0].get_ydata()[0]

    # find and move the corresponding text
    for txt in ax.texts:
        if "Market price p =" in txt.get_text():
            x_old, _ = txt.get_position()
            txt.set_position((x_old, y_price + 2.5))
            txt.set_va("bottom")
            break

plt.tight_layout()
plt.show()

## 10  Market value and self-cannibalisation (Formulas 22 and 23)

Prices differ between time steps, and a solar plant only produces in *some* of them. So the average price it actually earns is **not** the average price of the market. That is the **market value** (or capture price):

$$MV_v = \frac{\sum_t p_t \cdot g_{v,t} \cdot \Delta t}{\sum_t g_{v,t} \cdot \Delta t}
\qquad\qquad
VF_v = \frac{MV_v}{p^{\varnothing}}$$

The **value factor** `VF` is that market value divided by the plain average price. Above 1 means "produces when electricity is expensive". Below 1 means the opposite.

Now watch what happens below. We take **one and the same** PV plant, producing in t3, t4 and t5 (the hours around the demand peak), and we only change **how much** of it there is.

In [ ]:
PV_HOURS = ["t3", "t4", "t5"]   # our PV plant produces in these time steps only

def value_factor(pv_capacity, co2_price=CO2_BASE):
    """Market value and value factor of a PV plant that runs in PV_HOURS.
    The PV feed-in is included in the price calculation, so the plant
    affects the very prices it earns."""
    prices, gen = {}, {}
    for t, d in DEMAND.items():
        pv_now = pv_capacity if t in PV_HOURS else 0.0
        prices[t], _ = market_price(d, co2_price, wind=0.0, pv=pv_now)
        gen[t] = pv_now
    prices, gen = pd.Series(prices), pd.Series(gen)

    p_avg = prices.mean()                                   # all steps are equally long
    gen = pd.to_numeric(gen, errors="coerce").fillna(0.0)
    energy = float((gen * float(DT)).sum())
    if energy <= 0:            # a plant that produces nothing has no market value
        return prices, p_avg, float("nan"), float("nan")
    mv = (prices * gen * DT).sum() / energy                 # Formula 21
    return prices, p_avg, mv, mv / p_avg                    # Formula 22


for cap in [200, 1500]:
    prices, p_avg, mv, vf = value_factor(cap)
    print(f"PV capacity {cap:5.0f} MW")
    print(f"   prices        : {[f'{p:.2f}' for p in prices]}")
    print(f"   average price : {p_avg:.2f} €/MWh")
    print(f"   market value  : {mv:.2f} €/MWh")
    print(f"   value factor  : {vf:.2f}\n")

**Read that again.** Nothing about the technology changed. Same plant, same fleet, same production profile. Its value factor fell from **1.03 to 0.87** purely because there is **more of it**: with 1500 MW it pushes prices down in exactly the hours in which it earns money.

This is **self-cannibalisation**, and it is just the merit-order effect seen from the producer's side. It matters because what a renewable plant earns is its *market value*, not the average price. So a falling value factor erodes investment incentives even if average prices never move (Section 2.4.4).

The curve below traces the whole thing out.

In [ ]:
caps = np.arange(0, 2600, 50)
vfs = [value_factor(c)[3] for c in caps[1:]]

fig, ax = plt.subplots(figsize=(9.5, 5.0))
ax.plot(caps[1:], vfs, color="#08519c", linewidth=2.4, zorder=3)
ax.axhline(1.0, color="#666666", linestyle="--", linewidth=1.2, zorder=2)
ax.text(2500, 1.005, "value factor = 1", color="#666666", fontsize=9, ha="right", va="bottom")

for cap, note in [(200, "1.03"), (1500, "0.87")]:
    vf = value_factor(cap)[3]
    ax.plot([cap], [vf], marker="o", color="#e6550d", markersize=8, zorder=5)
    ax.annotate(
        f"{cap} MW\nVF = {note}", xy=(cap, vf), xytext=(cap, vf - 0.01), color="#e6550d", fontsize=9.5, ha="center", va="top")

ax.set_xlabel("Installed PV capacity (MW)", fontsize=10)
ax.set_ylabel("Value factor of PV (-)", fontsize=10)
ax.set_title("Self-cannibalisation (the more PV, the less each MWh of PV is worth)", fontsize=11)
ax.grid(False)
ax.set_axisbelow(True); ax.set_facecolor("white")

plt.tight_layout(); 
plt.show()

### Watch the capture price sink

The chart below is the same idea seen hour by hour. **Yellow bars** are the three time steps in which the PV plant produces, grey bars the ones in which it earns nothing. The dashed line is the market average, the dotted line is what the PV plant actually captures.

Pull the slider up from 0. At first the dotted line sits **above** the dashed one: PV produces at the peak, so it beats the average. Keep going and the yellow bars collapse under their own weight until the dotted line slips **below** the dashed one. The plant has cannibalised its own price.

For reference: in Germany in 2025 the real value factor of solar power was **0.50** (market value 45.08 against an average price of 89.32 €/MWh).

In [ ]:
def show_capture_price(pv_capacity=1500):
    prices, p_avg, mv, vf = value_factor(pv_capacity)

    fig, ax = plt.subplots(figsize=(9.5, 5.0))
    colours = ["#fdd835" if t in PV_HOURS else "#c8c8c8" for t in prices.index]
    ax.bar(prices.index, np.asarray(prices.to_numpy(), dtype=float), color=colours, edgecolor="white",
           linewidth=0.8, zorder=3)
    for t, p in prices.items():
        x = prices.index.get_loc(t)
        x = prices.index.tolist().index(t)
        ax.annotate(
            f"{p:.2f}",xy=(float(x), float(p)),xytext=(0, -4),textcoords="offset points",ha="center",va="top",fontsize=8.5,color="#555555",)

    ax.axhline(float(p_avg), color="#666666", linestyle="--", linewidth=1.6, zorder=4)
    ax.axhline(mv, color="#e6550d", linestyle=":", linewidth=2.2, zorder=4)
    ax.annotate(f"market average  {float(p_avg):.2f}", xy=(5.45, float(p_avg)), color="#666666",
                fontsize=9, va="bottom", ha="right")
    ax.annotate(f"PV capture price  {mv:.2f}", xy=(5.4, mv - 0.5), color="#e6550d",
                fontsize=9, va="top", ha="right")

    verdict = "above" if vf > 1 else "below"
    ax.set_title(f"{pv_capacity} MW PV capacity has a value factor of {vf:.2f} "
                 f"(PV earns {verdict} the market average)", fontsize=11)
    ax.set_ylabel("Market price (€/MWh)", fontsize=10)
    ax.set_xlabel("Time step (yellow for PV feed-in)", fontsize=10)
    ax.set_ylim(0, 90)
    ax.grid(axis="y", color="#ececec", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True); ax.set_facecolor("white")
    plt.tight_layout(); plt.show()

interactive(show_capture_price,
            static_example=dict(pv_capacity=1500),
            pv_capacity=(50, 2500, 50, 1500, "PV capacity (MW)"))

## 11  Market splitting under a transmission constraint (Figure 2-17)

So far the whole fleet competed in **one** merit order and cleared at **one** price. We now split the system into two zones, a cheap **North** and a more expensive **South**, connected by a transmission line whose commercial capacity is the net transfer capacity `NTC`. Total demand and the total fleet are unchanged; only their split between the zones is new (Table 2-14 and Table 2-15 of the Lehrbrief).

The dispatch rule has just one extra step compared with Section 4. First we dispatch the whole system as if the line were unlimited, which fixes how much the North would like to export. If that flow fits through the line, one uniform price clears both zones. If it exceeds the `NTC`, the flow is held at the limit, and **each zone then clears its own local merit order** against its local demand plus the export (North) or minus the import (South). The price difference that opens up is the pure result of the transmission limit, not of any difference in the plants.

Move the two sliders below. Watch the single merit order split into two, watch the two prices separate as you tighten the `NTC`, and find the value at which they merge again. Note that the line binds in five of the six time steps but **not** in t3, although t3 has the highest total demand: congestion follows the imbalance between the zones, not the level of demand.

In [ ]:
# --- Section 11: the two zones -------------------------------------------
# We reuse TECH, marginal_cost() and DT from the sections above and only add
# the zone assignment and the zonal demand (Table 2-14 and Table 2-15).

ZONE = pd.Series({
    "runOfRiver": "North", "nuclear": "North", "lignite": "North", "hardcoal_1": "North",
    "combinedCycle": "South", "gas_1": "South", "hardcoal_2": "South",
    "biomass": "South", "gas_2": "South",
})

ZONE_DEMAND = pd.DataFrame(
    {"North": [1400, 1600, 1900, 1750, 1800, 1600],
     "South": [1600, 1900, 2300, 2150, 2200, 2000]},
    index=[f"t{i+1}" for i in range(6)])

SOUTH_CAP = TECH["capacity"][ZONE == "South"].sum()   # 1720 MW


def zone_merit_order(zone, co2_price=CO2_BASE):
    """The merit order of one zone only - same idea as merit_order(), filtered by zone."""
    m = ZONE == zone
    r = pd.DataFrame({"label": TECH["label"][m], "capacity": TECH["capacity"][m].astype(float),
                      "mc": marginal_cost(co2_price)[m], "colour": TECH["colour"][m]})
    r = r.sort_values("mc", kind="stable")
    r["cum_before"] = r["capacity"].cumsum() - r["capacity"]
    r["cum_after"]  = r["capacity"].cumsum()
    return r


def zone_price(zone, demand, co2_price=CO2_BASE):
    """Zonal price = marginal cost of the cheapest plant in the zone with spare capacity."""
    mo = zone_merit_order(zone, co2_price)
    spare = mo[mo["cum_after"] > demand + 1e-9]
    if len(spare) == 0:
        return np.nan, "zone capacity exceeded"
    return spare.iloc[0]["mc"], spare.iloc[0]["label"]


def copper_flow(t, co2_price=CO2_BASE):
    """How much the North would export if the line were unlimited (the copper-plate flow)."""
    rows = pd.DataFrame({"cap": TECH["capacity"].astype(float),
                         "mc": marginal_cost(co2_price), "zone": ZONE})
    rows = rows.sort_values("mc", kind="stable")
    total = ZONE_DEMAND.loc[t].sum()
    served = north_gen = 0.0
    for _, r in rows.iterrows():
        take = min(r["cap"], max(0.0, total - served))
        served += take
        if r["zone"] == "North":
            north_gen += take
        if served >= total - 1e-9:
            break
    north_demand = ZONE_DEMAND["North"].astype("float64").loc[t]
    return float(north_gen) - north_demand


def two_zone_state(t, NTC, co2_price=CO2_BASE):
    """Return the dispatch of the two-zone market in time step t for a given NTC."""
    f0 = copper_flow(t, co2_price)
    dN, dS = ZONE_DEMAND.loc[t, "North"], ZONE_DEMAND.loc[t, "South"]
    congested = f0 > NTC + 1e-9
    flow = min(f0, NTC)
    adjN, adjS = dN + flow, dS - flow
    if adjS > SOUTH_CAP + 1e-9:                 # the South cannot be served -> infeasible
        return dict(feasible=False, flow=flow, f0=f0, min_ntc=dS - SOUTH_CAP)
    if not congested:
        psys, plant = zone_price("South", adjS, co2_price)
        return dict(feasible=True, congested=False, uniform=True, flow=f0,
                    adjN=dN, adjS=dS, pN=psys, pS=psys, plant=plant, f0=f0)
    pN, _ = zone_price("North", adjN, co2_price)
    pS, _ = zone_price("South", adjS, co2_price)
    return dict(feasible=True, congested=True, uniform=False, flow=flow,
                adjN=adjN, adjS=adjS, pN=pN, pS=pS, f0=f0)


# a quick check against the Lehrbrief numbers (NTC = 750 MW)
print("Copper-plate export of the North and the zonal prices at NTC = 750 MW:")
total_rent = 0.0
for t in ZONE_DEMAND.index:
    s = two_zone_state(t, 750)
    tag = "SPLIT " if s["congested"] else "uniform"
    if s["congested"]:
        pN_val = s.get("pN")
        if isinstance(pN_val, (int, float, np.integer, np.floating)):
            pN = float(pN_val)
        else:
            pN = float("nan")
        pS_val = s.get("pS")
        if isinstance(pS_val, (int, float, np.integer, np.floating)):
            pS = float(pS_val)
        else:
            pS = float("nan")
        flow_val = s.get("flow")
        if isinstance(flow_val, (int, float, np.integer, np.floating)):
            flow = float(flow_val)
        else:
            flow = 0.0
        rent = (pS - pN) * flow * float(DT)
    else:
        rent = 0.0
    total_rent += rent
    print(f"  {t}: export {s['f0']:5.0f} MW  ->  {tag}  "
          f"pN = {s['pN']:6.2f}  pS = {s['pS']:6.2f} €/MWh   rent {rent:8,.0f} €")
print(f"Total congestion rent over the day: {total_rent:,.0f} €")

In [ ]:
# --- Section 11: the interactive figure ----------------------------------

def _zone_panel(ax, zone, adj_demand, local_demand, price, flow, congested, co2_price):
    mo = zone_merit_order(zone, co2_price)
    ymax = max(95, mo["mc"].max() * 1.25)
    total_capacity = float(mo["cum_after"].max())
    xmax = max(total_capacity, float(local_demand), float(adj_demand))

    for _, r in mo.iterrows():
        ax.bar(r["cum_before"], r["mc"], width=r["capacity"], align="edge",
               color=r["colour"], edgecolor="white", linewidth=0.7,
               label=r["label"], zorder=3)

    gray, cflow, cprice = "#666666", "#c23b22", "#e6550d"

    # Local demand
    ax.axvline(local_demand, color=gray, linestyle="--", linewidth=1.3, zorder=4)
    ax.text(local_demand - 0.025 * xmax, ymax * 0.99, "local\ndemand",
            color=gray, fontsize=8, ha="right", va="top")

    # Demand after imports or exports
    if abs(flow) > 1e-9:
        ax.axvline(adj_demand, color=cflow, linestyle="-", linewidth=1.7, zorder=4)
        sign = "+" if zone == "North" else "\u2212"
        label_y = ymax * 0.80
        if np.isfinite(price) and abs(label_y - price) < 7:
            label_y = price - 6
        ax.text(adj_demand - 0.03 * xmax, label_y, f"demand\n{sign}{flow:.0f} MW",
                color=cflow, fontsize=8, ha="right", va="top")

    # Zonal price
    if np.isfinite(price):
        ax.plot([0, adj_demand], [price, price], color=cprice,
                linestyle=":", linewidth=1.8, zorder=4)
        price_label_x = min(0.45 * xmax, 0.70 * adj_demand)
        ax.text(price_label_x, price + 2, f"p = {price:.2f} €/MWh", color=cprice, fontsize=9,
                ha="center", va="bottom", zorder=5,
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=1))

    ax.set_xlim(0, xmax * 1.04)
    ax.set_ylim(0, ymax)
    ax.set_title(f"Zone {zone}", fontsize=10.5, fontweight="bold")
    ax.set_xlabel("Cumulative available capacity (MW)", fontsize=9)
    ax.legend(loc="upper left", fontsize=7.5, frameon=True)


def figure_2_17(t=5, NTC=750, co2_price=25):
    """Draw the two zonal merit orders for one time step and NTC."""
    tstep = f"t{t}"
    s = two_zone_state(tstep, NTC, co2_price)

    dN = ZONE_DEMAND.loc[tstep, "North"]
    dS = ZONE_DEMAND.loc[tstep, "South"]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)

    if not s["feasible"]:
        for ax in axes:
            ax.axis("off")

        axes[0].text(
            0.5, 0.5,
            f"NTC = {NTC} MW is infeasible in {tstep}.\n\n"
            f"The South has only {SOUTH_CAP:.0f} MW of capacity but must serve {dS:.0f} MW.\n"
            f"The smallest workable NTC here is {s['min_ntc']:.0f} MW.",
            ha="center", va="center", fontsize=11, color="#c23b22",
            transform=axes[0].transAxes
        )

        plt.tight_layout()
        plt.show()
        return

    _zone_panel(axes[0], "North", s["adjN"], dN, s["pN"],
                s["flow"], s["congested"], co2_price)
    _zone_panel(axes[1], "South", s["adjS"], dS, s["pS"],
                s["flow"], s["congested"], co2_price)

    axes[0].set_ylabel("Marginal cost, price (€/MWh)", fontsize=9)

    if s["congested"]:
        p_n_raw = s.get("pN", np.nan)
        p_s_raw = s.get("pS", np.nan)
        flow_raw = s.get("flow", 0.0)

        p_n = float(p_n_raw) if isinstance(
            p_n_raw, (int, float, np.integer, np.floating)) else np.nan
        p_s = float(p_s_raw) if isinstance(
            p_s_raw, (int, float, np.integer, np.floating)) else np.nan
        flow = float(flow_raw) if isinstance(
            flow_raw, (int, float, np.integer, np.floating)) else 0.0

        spread = p_s - p_n
        rent = spread * flow * float(DT)

        fig.suptitle(
            f"{tstep}  |  NTC = {NTC} MW  |  market split, flow at the limit "
            f"({flow:.0f} MW)  |  price spread {spread:.2f} €/MWh  |  "
            f"congestion rent {rent:,.0f} €",
            fontsize=10.5
        )
    else:
        fig.suptitle(
            f"{tstep}  |  NTC = {NTC} MW  |  line not congested, single price "
            f"{s['pN']:.2f} €/MWh  |  desired flow {s['f0']:.0f} MW fits",
            fontsize=10.5
        )

    plt.tight_layout()
    plt.show()


# Keep the carbon price fixed at the base-model value of 25 EUR/t
def figure_2_17_controls(t=5, NTC=750):
    figure_2_17(t=t, NTC=NTC, co2_price=25)


interactive(
    figure_2_17_controls,
    static_example=dict(t=5, NTC=750),
    t=(1, 6, 1, 5, "time step t"),
    NTC=(300, 2000, 50, 750, "NTC (MW)")
)

## 12  Long-term investment: screening curves and the optimal capacity mix (Figure 2-20)

Sections 1 to 11 took the plant fleet as given. We now ask which fleet a cost-minimising investor would build. The idea of **screening curves** is simple: for one MW of a technology, the total annual cost is its annualised investment cost plus its variable cost multiplied by the number of hours that MW actually runs. That is a straight line in the full load hours, with the annualised investment cost as intercept and the variable cost as slope. For any level of utilisation the cheapest technology is the one whose line lies lowest, so the **lower envelope** of all lines tells us what to build.

The annualised investment costs come from Table 2-16 of the teaching scipt; they were obtained by applying the annuity factor of Section 2.2.3 to the CAPEX, lifetime and fixed OPEX of Table 2-5 at a real WACC of 6 per cent.

Unserved load is treated as one more option, with an intercept of zero and a slope equal to the **value of lost load**. It wins for very short durations, which is why an economically optimal system sheds load in a small number of hours instead of building a turbine for them.

The lower panel maps the envelope onto the **annual load duration curve** (Table 2-17): each horizontal slice of the curve is one MW of capacity and the width of the slice is the number of hours it runs.

Move the two sliders. Lower the carbon price and find the value at which lignite takes over the base of the curve. Then raise the value of lost load and find the point at which the model stops shedding load altogether.

In [ ]:
CANDIDATES = pd.DataFrame({
    "label":  ["Nuclear", "Lignite", "Hard coal", "Gas CCGT", "Gas OCGT"],
    "c_inv":  [352_330.0, 186_550.0, 182_300.0, 92_730.0, 42_120.0],
    "c_var0": [6.06, 12.25, 32.15, 34.11, 67.50],
    "ef":     [0.00, 1.05, 0.80, 0.35, 0.50],
    "colour": ["#c724b1", "#7b5141", "#333333", "#fdae6b", "#e6550d"],
}, index=["nuclear", "lignite", "hardCoal", "combinedCycle", "openCycle"])

# annual load duration curve, sorted descending (Table 2-17)
LDC_LOAD = np.array([4400, 4200, 4000, 3900, 3600, 3500, 3000, 2600], dtype=float)
LDC_DUR  = np.array([   3,   97,  400,  700, 1800, 2000, 2400, 1360], dtype=float)
LDC_CUM  = np.cumsum(LDC_DUR)          # hours per year with load >= this level
SHED_COLOUR = "#c23b22"


def candidate_costs(co2_price=CO2_BASE, VOLL=10_000.0):
    """Return the list of (label, intercept, slope, colour) lines, shedding included."""
    lines = [(r["label"], r["c_inv"], r["c_var0"] + r["ef"] * co2_price, r["colour"])
             for _, r in CANDIDATES.iterrows()]
    lines.append(("Load shedding", 0.0, float(VOLL), SHED_COLOUR))
    return lines


def cheapest_at(hours, lines):
    """Which option is on the lower envelope at a given number of full load hours?"""
    return min(lines, key=lambda L: L[1] + L[2] * hours)


def capacity_blocks(co2_price=CO2_BASE, VOLL=10_000.0):
    """Split the load duration curve into capacity increments and assign a technology."""
    lines = candidate_costs(co2_price, VOLL)
    blocks = []                                   # (label, width MW, duration h, colour)
    for k in range(len(LDC_LOAD) - 1):            # increments from the top downwards
        lab, _, _, col = cheapest_at(LDC_CUM[k], lines)
        blocks.append((lab, LDC_LOAD[k] - LDC_LOAD[k + 1], LDC_CUM[k], col))
    lab, _, _, col = cheapest_at(LDC_CUM[-1], lines)
    blocks.append((lab, LDC_LOAD[-1], LDC_CUM[-1], col))     # the base of the curve
    return blocks


def optimal_mix(co2_price=CO2_BASE, VOLL=10_000.0):
    """Aggregate the capacity increments into a capacity per technology (MW)."""
    mix = {}
    for lab, width, _, _ in capacity_blocks(co2_price, VOLL):
        mix[lab] = mix.get(lab, 0.0) + width
    return mix


# a quick check against the Lehrbrief numbers
for p in (25, 10, 60):
    mix = optimal_mix(p, 10_000.0)
    print(f"CO2 = {p:3d} EUR/t  ->  " +
          ", ".join(f"{k}: {v:,.0f} MW" for k, v in mix.items()))

In [ ]:
from matplotlib.patches import Rectangle      # used for the capacity bands

def figure_2_20(co2_price=25, VOLL=10000):
    lines  = candidate_costs(co2_price, VOLL)
    blocks = capacity_blocks(co2_price, VOLL)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10.2, 7.6), sharex=True,
                                   gridspec_kw={"height_ratios": [1.12, 1.0], "hspace": 0.12})

    # ---- upper panel: the screening curves and their lower envelope
    hs = np.linspace(0, 8760, 1500)
    for lab, ci, cv, col in lines:
        style = dict(ls=(0, (5, 3)), lw=1.6) if lab == "Load shedding" else dict(lw=2.0)
        ax1.plot(hs, (ci + cv * hs) / 1000, color=col, label=lab, zorder=3, **style)
    env = np.array([min(ci + cv * x for _, ci, cv, _ in lines) for x in hs])
    ax1.plot(hs, env / 1000, color="#111111", lw=3.4, alpha=.28, zorder=2)

    # mark the durations at which the option on the envelope changes
    switches, prev = [], cheapest_at(hs[0], lines)[0]
    for x in hs[1:]:
        cur = cheapest_at(x, lines)[0]
        if cur != prev:
            switches.append(x); prev = cur
    ytop = max(CANDIDATES["c_inv"].max() * 2.1, 740_000) / 1000
    for xv in switches:
        for ax in (ax1, ax2):
            ax.axvline(xv, color="#555555", ls=":", lw=1.0, zorder=1)
        ax1.text(xv + 95, ytop * 0.94, f"{xv:,.0f} h", fontsize=8, color="#444444")

    ax1.set_xlim(0, 8760); ax1.set_ylim(0, ytop)
    ax1.set_ylabel("Annual cost per MW\n(thousand EUR per MW and year)", fontsize=9)
    ax1.legend(loc="upper left", fontsize=8, frameon=True, ncol=2)
    ax1.set_title(f"Screening curves at {co2_price} EUR/tCO\u2082 and a VOLL of {VOLL:,.0f} EUR/MWh",
                  fontsize=10.5, fontweight="bold")

    # ---- lower panel: load duration curve with the resulting capacity bands
    xs, ys, x = [], [], 0.0
    for load, dur in zip(LDC_LOAD, LDC_DUR):
        xs += [x, x + dur]; ys += [load, load]; x += dur

    bottom = 0.0
    for lab, width, dur, col in sorted(blocks, key=lambda b: -b[2]):   # widest block first
        ax2.add_patch(Rectangle((0, bottom), dur, width, fc=col, alpha=.55, ec="none", zorder=1))
        bottom += width
    ax2.plot(xs, ys, color="#00449e", lw=2.4, zorder=4, label="Load duration curve")

    ax2.set_xlim(0, 8760); ax2.set_ylim(0, LDC_LOAD.max() * 1.14)
    ax2.set_xlabel("Hours of the year with a load of at least this level (h per year)", fontsize=9)
    ax2.set_ylabel("Load, installed capacity (MW)", fontsize=9)
    ax2.legend(loc="upper right", fontsize=8, frameon=True)

    mix   = optimal_mix(co2_price, VOLL)
    built = ", ".join(f"{k} {v:,.0f} MW" for k, v in mix.items() if k != "Load shedding")
    shed  = mix.get("Load shedding", 0.0)
    lole  = LDC_CUM[0] if shed > 0 else 0.0
    ax2.set_title(f"Optimal mix: {built}   |   unserved {shed:,.0f} MW, LOLE {lole:,.0f} h/a",
                  fontsize=10.5, fontweight="bold")
    plt.show()


interactive(figure_2_20,
            static_example=dict(co2_price=25, VOLL=10000),
            co2_price=(0, 150, 5, 25, "CO\u2082 price (\u20ac/t)"),
            VOLL=(1000, 30000, 1000, 10000, "Value of lost load (\u20ac/MWh)"))